# Band-edge spacing boundary after the first adjacent-loop test

A companion notebook for `notes/band-edge-spacing-boundary.md`.

This keeps the same bounded loop as the previous adjacent-channel note and moves only one knob: channel spacing at fixed loop gain and fixed `0 dB` adjacent power.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPTS = REPO / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from waveform_carrier_front_ends import study_band_edge_closed_loop_spacing_sweep


In [ ]:
spacings = [round(0.80 + 0.01 * idx, 2) for idx in range(86)]
rows = study_band_edge_closed_loop_spacing_sweep(spacings, adjacent_relative_power_db=0.0)
rows[:4]


In [ ]:
lookup = {(row.design, round(row.channel_spacing, 2)): row for row in rows}
settle_spacing = next(spacing for spacing in spacings if lookup[('gnuradio_half_sine', spacing)].tail_within_threshold_fraction == 1.0)
crossover_spacing = next(
    spacing
    for spacing in spacings
    if lookup[('gnuradio_half_sine', spacing)].tail_mean_abs_residual_cfo <= lookup[('proxy_bandpass', spacing)].tail_mean_abs_residual_cfo
)
settle_spacing, crossover_spacing


In [ ]:
summary = []
for spacing in [1.00, settle_spacing, crossover_spacing]:
    proxy = lookup[('proxy_bandpass', round(spacing, 2))]
    half = lookup[('gnuradio_half_sine', round(spacing, 2))]
    summary.append({
        'spacing': round(spacing, 2),
        'proxy_mean_tail_residual': round(proxy.tail_mean_abs_residual_cfo, 4),
        'proxy_tail_fraction': round(proxy.tail_within_threshold_fraction, 3),
        'half_sine_mean_tail_residual': round(half.tail_mean_abs_residual_cfo, 4),
        'half_sine_tail_fraction': round(half.tail_within_threshold_fraction, 3),
    })
summary


## Readout

This sweep leaves one durable teaching point: the phrase *when does the preference flip?* is metric-dependent.

- The half-sine lane gets fully back inside the `±0.05 R_s` settle band at about `1.24 R_s`.
- It does **not** beat the proxy on mean tail residual until about `1.57 R_s`.

That is the sharper boundary the queue was missing.